# Behavioral Reliability of Explainable Intrusion Detection Under Distribution Shift
## Production Research Pipeline Engine — Google Drive Persistent Storage Edition

This notebook contains the final, end-to-end experimental pipeline mapped to a persistent Google Drive paths. It natively addresses data leakage, drops rare structural outliers (`Heartbleed` / `Web_SQL_Injection`), and isolates `Web_Brute_Force` as a strict zero-day vector.

In [ ]:
# BLOCK 1: MOUNT GOOGLE DRIVE & ATTACH DEPENDENCY RUNTIMES
from google.colab import drive
drive.mount('/content/drive')

!pip install shap lightgbm scikit-learn psutil pandas numpy --quiet

import os
import re
import gc
import time
import json
import glob
import psutil
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score, brier_score_loss
from sklearn.calibration import IsotonicRegression
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
RESULTS_DIR = "./behavioral_reliability_perfected_results"
os.makedirs(os.path.join(RESULTS_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)

# Central Configuration Object mapped to your exact Google Drive path names
CONFIG = {
    'base_path': '/content/drive/MyDrive',
    'bccc_dir': '/content/drive/MyDrive/BCCC-CIC-IDS-2017',
    'cic2017_dir': '/content/drive/MyDrive/CIC-IDS2017',
    'cse2018_dir': '/content/drive/MyDrive/CSE-CIC-IDS2018',
    'results_dir': '/content/drive/MyDrive/IDS_XAI_Project/',
    'keep_benign_files': {'friday_benign.csv', 'thursday_benign.csv'},
    'min_class_count': 30,
    'rare_classes_expected': {'Heartbleed', 'Web_SQL_Injection'},
    'zero_day_class': 'Web_Brute_Force',
    'max_rows_per_class': 500
}
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print("[Initialization] Virtual execution environments established and dependencies fixed.")

In [ ]:
# BLOCK 2: GLOBAL SYSTEM SCHEMAS, SCHEMA TRANSLATORS, AND LATIN-1 CLEANERS
GLOBAL_CLASS_INDEX = {
    'Benign': 0, 'Botnet_ARES': 1, 'DDoS_LOIT': 2, 'DoS_GoldenEye': 3,
    'DoS_Hulk': 4, 'DoS_Slowhttptest': 5, 'DoS_Slowloris': 6, 'FTP-Patator': 7,
    'Port_Scan': 8, 'SSH-Patator': 9, 'Web_Brute_Force': 10, 'Web_XSS': 11
}
REVERSE_CLASS_INDEX = {v: k for k, v in GLOBAL_CLASS_INDEX.items()}

CANONICAL_LABEL_MAP = {
    'BENIGN': 'Benign', 'Benign': 'Benign', 'benign': 'Benign',
    'Bot': 'Botnet_ARES', 'Botnet_ARES': 'Botnet_ARES', 'BotnetARES': 'Botnet_ARES',
    'DDoS': 'DDoS_LOIT', 'DDOS': 'DDoS_LOIT', 'DDoS_LOIT': 'DDoS_LOIT',
    'DoS_Hulk': 'DoS_Hulk', 'Hulk': 'DoS_Hulk', 'DoS_attacks_Hulk': 'DoS_Hulk', 'DoS attacks-Hulk': 'DoS_Hulk',
    'DoS_GoldenEye': 'DoS_GoldenEye', 'DoS_Golden_Eye': 'DoS_GoldenEye', 'DoS attacks-GoldenEye': 'DoS_GoldenEye', 'DoS_attacks_GoldenEye': 'DoS_GoldenEye',
    'DoS_slowloris': 'DoS_Slowloris', 'DoS_Slowloris': 'DoS_Slowloris', 'DoS attacks-Slowloris': 'DoS_Slowloris', 'DoS_attacks_Slowloris': 'DoS_Slowloris',
    'DoS_Slowhttptest': 'DoS_Slowhttptest', 'DoS_slowhttptest': 'DoS_Slowhttptest', 'DoS attacks-SlowHTTPTest': 'DoS_Slowhttptest', 'DoS_attacks_SlowHTTPTest': 'DoS_Slowhttptest',
    'FTP_Patator': 'FTP-Patator', 'FTP-Patator': 'FTP-Patator', 'FTPPatator': 'FTP-Patator', 'FTP-BruteForce': 'FTP-Patator', 'FTP_BruteForce': 'FTP-Patator',
    'SSH_Patator': 'SSH-Patator', 'SSH-Patator': 'SSH-Patator', 'SSHPatator': 'SSH-Patator', 'SSH-Bruteforce': 'SSH-Patator', 'SSH_Bruteforce': 'SSH-Patator',
    'PortScan': 'Port_Scan', 'Port_Scan': 'Port_Scan', 'Portscan': 'Port_Scan',
    'Web_Attack_Brute_Force': 'Web_Brute_Force', 'Web_Brute_Force': 'Web_Brute_Force', 'Brute Force -Web': 'Web_Brute_Force', 'Brute_Force_Web': 'Web_Brute_Force',
    'Web_Attack_XSS': 'Web_XSS', 'Web_XSS': 'Web_XSS', 'Brute Force -XSS': 'Web_XSS', 'Brute_Force_XSS': 'Web_XSS',
    'Web_Attack_Sql_Injection': 'Web_XSS', 'Web_SQL_Injection': 'Web_XSS', 'SQL Injection': 'Web_XSS', 'SQL_Injection': 'Web_XSS',
    "Web_Attack__Brute_Force": "Web_Brute_Force",
    "Web_Attack__XSS": "Web_XSS",
    "Web_Attack__Sql_Injection": "Web_XSS"
}

# --- PRIMARY TRANSLATOR: RE-MAPPING INTERNAL INVARIANT FEATURES TO RAW CIC-IDS2017 DISK HEADERS ---
# Aligned directly with your explicit file headers from your Google Drive audit logs
FEATURE_MAP_CIC = {
    'protocol': ' Protocol',
    'duration': ' Flow Duration',
    'packets_count': ' Flow Packets/s',
    'fwd_packets_count': ' Total Fwd Packets',
    'bwd_packets_count': ' Total Backward Packets',
    'fwd_total_payload_bytes': 'Total Length of Fwd Packets',
    'bwd_total_payload_bytes': ' Total Length of Bwd Packets',
    'fwd_payload_bytes_max': ' Fwd Packet Length Max',
    'fwd_payload_bytes_min': ' Fwd Packet Length Min',
    'fwd_payload_bytes_mean': ' Fwd Packet Length Mean',
    'fwd_payload_bytes_std': ' Fwd Packet Length Std',
    'bwd_payload_bytes_max': 'Bwd Packet Length Max',
    'bwd_payload_bytes_min': ' Bwd Packet Length Min',
    'bwd_payload_bytes_mean': ' Bwd Packet Length Mean',
    'bwd_payload_bytes_std': ' Bwd Packet Length Std',
    'bytes_rate': 'Flow Bytes/s',
    'packets_rate': ' Flow Packets/s',
    'packets_iat_mean': ' Flow IAT Mean',
    'packet_iat_std': ' Flow IAT Std',
    'packet_iat_max': ' Flow IAT Max',
    'packet_iat_min': ' Flow IAT Min',
    'fwd_packets_iat_total': 'Fwd IAT Total',
    'fwd_packets_iat_mean': ' Fwd IAT Mean',
    'fwd_packets_iat_std': ' Fwd IAT Std',
    'fwd_packets_iat_max': ' Fwd IAT Max',
    'fwd_packets_iat_min': ' Fwd IAT Min',
    'bwd_packets_iat_total': 'Bwd IAT Total',
    'bwd_packets_iat_mean': ' Bwd IAT Mean',
    'bwd_packets_iat_std': ' Bwd IAT Std',
    'bwd_packets_iat_max': ' Bwd IAT Max',
    'bwd_packets_iat_min': ' Bwd IAT Min',
    'fwd_psh_flag_counts': 'Fwd PSH Flags',
    'bwd_psh_flag_counts': ' Bwd PSH Flags',
    'fwd_urg_flag_counts': ' Fwd URG Flags',
    'bwd_urg_flag_counts': ' Bwd URG Flags',
    'fwd_total_header_bytes': ' Fwd Header Length',
    'bwd_total_header_bytes': ' Bwd Header Length',
    'fwd_packets_rate': 'Fwd Packets/s',
    'bwd_packets_rate': ' Bwd Packets/s',
    'payload_bytes_min': ' Min Packet Length',
    'payload_bytes_max': ' Max Packet Length',
    'payload_bytes_mean': ' Packet Length Mean',
    'payload_bytes_std': ' Packet Length Std',
    'payload_bytes_variance': ' Packet Length Variance',
    'fin_flag_counts': 'FIN Flag Count',
    'syn_flag_counts': ' SYN Flag Count',
    'rst_flag_counts': ' RST Flag Count',
    'psh_flag_counts': ' PSH Flag Count',
    'ack_flag_counts': ' ACK Flag Count',
    'urg_flag_counts': ' URG Flag Count',
    'cwr_flag_counts': ' CWE Flag Count',
    'ece_flag_counts': ' ECE Flag Count',
    'down_up_rate': ' Down/Up Ratio',
    'fwd_avg_segment_size': ' Avg Fwd Segment Size',
    'bwd_avg_segment_size': ' Avg Bwd Segment Size',
    'avg_segment_size': ' Average Packet Size',
    'avg_fwd_bytes_per_bulk': 'Fwd Avg Bytes/Bulk',
    'avg_fwd_packets_per_bulk': ' Fwd Avg Packets/Bulk',
    'avg_fwd_bulk_rate': ' Fwd Avg Bulk Rate',
    'avg_bwd_bytes_per_bulk': ' Bwd Avg Bytes/Bulk',
    'avg_bwd_packets_bulk_rate': ' Bwd Avg Packets/Bulk',
    'avg_bwd_bulk_rate': 'Bwd Avg Bulk Rate',
    'subflow_fwd_packets': 'Subflow Fwd Packets',
    'subflow_fwd_bytes': ' Total Length of Fwd Packets', # Maps cleanly to raw capacity limits
    'subflow_bwd_packets': ' Subflow Bwd Packets',
    'subflow_bwd_bytes': ' Subflow Bwd Bytes',
    'fwd_init_win_bytes': 'Init_Win_bytes_forward',
    'bwd_init_win_bytes': ' Init_Win_bytes_backward',
    'active_mean': 'Active Mean',
    'active_std': ' Active Std',
    'active_max': ' Active Max',
    'active_min': ' Active Min',
    'idle_mean': 'Idle Mean',
    'idle_std': ' Idle Std',
    'idle_max': ' Idle Max',
    'idle_min': ' Idle Min'
}

# --- CROSS-DATASET TRANSLATOR: RE-MAPPING INTERNAL INVARIANT FEATURES TO RAW CSE-CIC-IDS2018 DISK HEADERS ---
# Aligned directly with your explicit file headers from your Google Drive audit logs
FEATURE_MAP_CSE2018 = {
    'protocol': 'Protocol',
    'duration': 'Flow Duration',
    'packets_count': 'Flow Pkts/s',
    'fwd_packets_count': 'Tot Fwd Pkts',
    'bwd_packets_count': 'Tot Bwd Pkts',
    'fwd_total_payload_bytes': 'TotLen Fwd Pkts',
    'bwd_total_payload_bytes': 'TotLen Bwd Pkts',
    'fwd_payload_bytes_max': 'Fwd Pkt Len Max',
    'fwd_payload_bytes_min': 'Fwd Pkt Len Min',
    'fwd_payload_bytes_mean': 'Fwd Pkt Len Mean',
    'fwd_payload_bytes_std': 'Fwd Pkt Len Std',
    'bwd_payload_bytes_max': 'Bwd Pkt Len Max',
    'bwd_payload_bytes_min': 'Bwd Pkt Len Min',
    'bwd_payload_bytes_mean': 'Bwd Pkt Len Mean',
    'bwd_payload_bytes_std': 'Bwd Pkt Len Std',
    'bytes_rate': 'Flow Byts/s',
    'packets_rate': 'Flow Pkts/s',
    'packets_iat_mean': 'Flow IAT Mean',
    'packet_iat_std': 'Flow IAT Std',
    'packet_iat_max': 'Flow IAT Max',
    'packet_iat_min': 'Flow IAT Min',
    'fwd_packets_iat_total': 'Fwd IAT Tot',
    'fwd_packets_iat_mean': 'Fwd IAT Mean',
    'fwd_packets_iat_std': 'Fwd IAT Std',
    'fwd_packets_iat_max': 'Fwd IAT Max',
    'fwd_packets_iat_min': 'Fwd IAT Min',
    'bwd_packets_iat_total': 'Bwd IAT Tot',
    'bwd_packets_iat_mean': 'Bwd IAT Mean',
    'bwd_packets_iat_std': 'Bwd IAT Std',
    'bwd_packets_iat_max': 'Bwd IAT Max',
    'bwd_packets_iat_min': 'Bwd IAT Min',
    'fwd_psh_flag_counts': 'Fwd PSH Flags',
    'bwd_psh_flag_counts': 'Bwd PSH Flags',
    'fwd_urg_flag_counts': 'Fwd URG Flags',
    'bwd_urg_flag_counts': 'Bwd URG Flags',
    'fwd_total_header_bytes': 'Fwd Header Len',
    'bwd_total_header_bytes': 'Bwd Header Len',
    'fwd_packets_rate': 'Fwd Pkts/s',
    'bwd_packets_rate': 'Bwd Pkts/s',
    'payload_bytes_min': 'Pkt Len Min',
    'payload_bytes_max': 'Pkt Len Max',
    'payload_bytes_mean': 'Pkt Len Mean',
    'payload_bytes_std': 'Pkt Len Std',
    'payload_bytes_variance': 'Pkt Len Var',
    'fin_flag_counts': 'FIN Flag Cnt',
    'syn_flag_counts': 'SYN Flag Cnt',
    'rst_flag_counts': 'RST Flag Cnt',
    'psh_flag_counts': 'PSH Flag Cnt',
    'ack_flag_counts': 'ACK Flag Cnt',
    'urg_flag_counts': 'URG Flag Cnt',
    'cwr_flag_counts': 'CWE Flag Count',
    'ece_flag_counts': 'ECE Flag Cnt',
    'down_up_rate': 'Down/Up Ratio',
    'fwd_avg_segment_size': 'Fwd Seg Size Avg',
    'bwd_avg_segment_size': 'Bwd Seg Size Avg',
    'avg_segment_size': 'Pkt Size Avg',
    'avg_fwd_bytes_per_bulk': 'Fwd Byts/b Avg',
    'avg_fwd_packets_per_bulk': 'Fwd Pkts/b Avg',
    'avg_fwd_bulk_rate': 'Fwd Blk Rate Avg',
    'avg_bwd_bytes_per_bulk': 'Bwd Byts/b Avg',
    'avg_bwd_packets_bulk_rate': 'Bwd Pkts/b Avg',
    'avg_bwd_bulk_rate': 'Bwd Blk Rate Avg',
    'subflow_fwd_packets': 'Subflow Fwd Pkts',
    'subflow_fwd_bytes': 'Subflow Fwd Byts',
    'subflow_bwd_packets': 'Subflow Bwd Pkts',
    'subflow_bwd_bytes': 'Subflow Bwd Byts',
    'fwd_init_win_bytes': 'Init Fwd Win Byts',
    'bwd_init_win_bytes': 'Init Bwd Win Byts',
    'active_mean': 'Active Mean',
    'active_std': 'Active Std',
    'active_max': 'Active Max',
    'active_min': 'Active Min',
    'idle_mean': 'Idle Mean',
    'idle_std': 'Active Std', # Correlated mapping fallback layer
    'idle_max': 'Idle Max',
    'idle_min': 'Idle Min'
}

print("[Initialization] Configuration global maps and cross-dataset schema translation profiles securely locked.")

In [ ]:
# BLOCK 3: STRATIFIED PIPELINE INGESTION CORE (RARE CLASS FILTERS REMOVAL ENFORCED)
def clean_colname(c):
    c = str(c).strip().replace('\ufeff', '')
    c = re.sub(r'[^0-9a-zA-Z]+', '_', c)
    c = re.sub(r'_+', '_', c).strip('_').lower()
    return c

def normalize_columns(df):
    df.columns = [clean_colname(c) for c in df.columns]
    return df

def standardize_label_text(x):
    if pd.isna(x): return "Benign"
    s = str(x).strip().replace('–', '-').replace('—', '-').replace(' ', '_').replace('-', '_')
    s = re.sub(r'_+', '_', s)
    return s

def normalize_labels(df):
    candidates = ['label', 'class', 'attack', 'attack_cat', 'category']
    lbl_col = None
    for c in candidates:
        if c in df.columns:
            lbl_col = c
            break
    if lbl_col is None:
        df['label'] = 'Benign'
        return df
    df.rename(columns={lbl_col: 'label'}, inplace=True)
    df['label'] = df['label'].astype(str).str.strip().map(
        lambda x: CANONICAL_LABEL_MAP.get(x, CANONICAL_LABEL_MAP.get(standardize_label_text(x), 'Benign'))
    )
    return df

def load_bccc_balanced_workspace():
    print(f"[Ingestion Loop] Aggregating matrices inside path: {CONFIG['bccc_dir']}")
    csv_files = sorted(glob.glob(os.path.join(CONFIG['bccc_dir'], "**", "*.csv"), recursive=True))

    if len(csv_files) == 0:
        print("[Ingestion Fallback] Target folder empty. Deploying standalone matrix simulation...")
        np.random.seed(RANDOM_STATE)
        n_samples = 35000
        features = list(FEATURE_MAP_CIC.values()) + ['active_mean', 'active_max', 'idle_mean', 'idle_max']
        data = {f: np.random.exponential(scale=15.0, size=n_samples) for f in features}
        data['src_ip'] = np.random.choice([f"192.168.10.{i}" for i in range(1, 150)], size=n_samples)
        data['label'] = np.random.choice(list(GLOBAL_CLASS_INDEX.keys()), size=n_samples, p=[0.3] + [0.7/11]*11)
        df = pd.DataFrame(data)
        # Enforce dynamic rare class extraction drop on synthetic generation too
        df = df[~df['label'].isin(CONFIG['rare_classes_expected'])]
        return df, features

    parts = []
    for f in csv_files:
        fname = os.path.basename(f).lower()
        # Native class imbalance control layer execution
        if 'benign' in fname and fname not in CONFIG['keep_benign_files']:
            continue
        try:
            df = pd.read_csv(f, nrows=12000, low_memory=False)
            df = normalize_columns(df)
            df = normalize_labels(df)

            # CRITICAL CORRECTION: Explicitly exclude rare classes below min threshold count limits
            df = df[~df['label'].isin(CONFIG['rare_classes_expected'])]
            parts.append(df)
        except Exception as e:
            continue

    master_df = pd.concat(parts, ignore_index=True, sort=False)

    # Secondary programmatic safe verification trap
    for rare_cls in CONFIG['rare_classes_expected']:
        master_df = master_df[master_df['label'] != rare_cls]

    features = [c for c in master_df.columns if c != 'label' and c not in ['src_ip', 'dst_ip', 'timestamp']]
    return master_df, features

df_raw, original_features = load_bccc_balanced_workspace()
print("[Ingestion Engine] Historical analytical core shape (Outliers Dropped):", df_raw.shape)
print("[Ingestion Engine] Current Class Populations Summary:\n", df_raw['label'].value_counts())

In [ ]:
# BLOCK 4: STAGE 1 — MEMORIZATION AND TOPOLOGY LEAKAGE ELIMINATION
def behavior_focused_clean(df, continuous_features):
    print("[Stage 1] Evaporating routing metadata fields to enforce behavioral splits...")

    # Clean out extreme value floats from target structures
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=['label'])

    # FIXED: True exact administrative, contextual, and tracking leakage columns
    leakage_cols = [
        "flow_id", "timestamp", "Date", "src_ip", "dst_ip",
        "HostID", "label", "source_file", "src_port", "dst_port"
    ]

    # Convert leakage elements to lowercase for robust case-insensitive alignment protection
    leakage_lower = [str(col).lower() for col in leakage_cols]

    # Isolate valid, invariant behavioral features while systematically scrubbing leakage channels
    valid_features = [
        f for f in continuous_features
        if f in df.columns and str(f).lower() not in leakage_lower
    ]

    # Enforce numeric integrity and impute missing frames using baseline medians
    for col in valid_features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Handle cases where columns might be empty or all-NaN gracefully
        if df[col].notna().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(0.0)

    return df, valid_features

# Execute clean, data-leakage-free sanitization across the active dataframe workspace
df_clean, expected_features = behavior_focused_clean(df_raw, original_features)
print(f"[Stage 1 Complete] Invariant feature attributes isolated count: {len(expected_features)}")

In [ ]:
# BLOCK 5: STAGE 2 — COVARIATE DRIFT HOST RATIO ABLATION STUDY & AUTO-SELECTION ENGINE
def run_host_ablation_study(df, expected_features):
    print("=" * 115)
    print("[Stage 2] Launching Comprehensive Topological Host-Ratio Ablation Study Engine...")
    print("=" * 115)

    # Isolate exact host tracking key from the source IP column metrics
    df = df.copy()
    df['hostid_str'] = df['src_ip'].astype(str).map(lambda x: x.split('.')[-1] if '.' in x else '1')
    unique_hosts = df['hostid_str'].unique()

    # Establish total multi-class global baseline target pool size
    global_total_labels = df['label'].nunique()

    # Stage 0: Initial Scenario-Wise Base Data Assessment Printing
    print("\nAfter scenario-wise time split:")
    print(f"Train rows: {len(df[df['label'] != CONFIG['zero_day_class']]):,} Labels: {df[df['label'] != CONFIG['zero_day_class']]['label'].nunique()}")
    print(f"Validation base rows: {int(len(df)*0.2):,} Labels: {global_total_labels}")
    print(f"Test base rows: {int(len(df)*0.2):,} Labels: {global_total_labels}\n")

    ratios = [0.7, 0.6, 0.5, 0.4]
    ablation_records = []
    partition_caches = {}

    for r in ratios:
        np.random.seed(RANDOM_STATE)
        seen_hosts = np.random.choice(unique_hosts, size=int(len(unique_hosts) * r), replace=False)

        # Segment data records based on host placement pools
        df_seen = df[df['hostid_str'].isin(seen_hosts)].copy()
        df_unseen = df[~df['hostid_str'].isin(seen_hosts)].copy()

        # Enforce quarantine matrix splitting matching production conditions
        df_train_p = df_seen.sample(frac=0.6, random_state=RANDOM_STATE)
        df_val_p = df_unseen.sample(frac=0.2, random_state=RANDOM_STATE)
        df_host_p = df_unseen.sample(frac=0.2, random_state=RANDOM_STATE)
        df_time_p = df_seen.sample(frac=0.2, random_state=RANDOM_STATE)
        df_hard_p = df_unseen.sample(frac=0.2, random_state=RANDOM_STATE)

        # Calculate label deficiencies against absolute training targets
        val_loss = max(0, global_total_labels - df_val_p['label'].nunique())
        time_loss = max(0, global_total_labels - df_time_p['label'].nunique())

        rec = {
            'Seen Ratio': r,
            'Unseen Ratio': round(1.0 - r, 1),
            'Val Rows': len(df_val_p), 'Val Hosts': df_val_p['hostid_str'].nunique(), 'Val Labels': df_val_p['label'].nunique(),
            'Host Rows': len(df_host_p), 'Host Hosts': df_host_p['hostid_str'].nunique(), 'Host Labels': df_host_p['label'].nunique(),
            'Time Rows': len(df_time_p), 'Time Hosts': df_time_p['hostid_str'].nunique(), 'Time Labels': df_time_p['label'].nunique(),
            'Hard Rows': len(df_hard_p), 'Hard Hosts': df_hard_p['hostid_str'].nunique(), 'Hard Labels': df_hard_p['label'].nunique(),
            'Val Label Loss': val_loss,
            'Time Label Loss': time_loss
        }
        ablation_records.append(rec)

        # Cache full data splits internally for final class summary rendering
        partition_caches[r] = (df_train_p, df_val_p, df_host_p, df_time_p, df_hard_p)

    ablation_df = pd.DataFrame(ablation_records)

    # Display Primary Experiment Summary Block
    print("Host split ratio experiment summary:")
    print(ablation_df.drop(columns=['Val Label Loss', 'Time Label Loss']).to_string(index=False))
    print("\n" + "-"*50 + "\n")

    # Sort results according to safety criteria: Prioritize lowest label loss, then maximize validation row volumes
    sorted_ablation_df = ablation_df.sort_values(
        by=['Val Label Loss', 'Time Label Loss', 'Val Rows'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # Display Sorted Comparison Block
    print("Sorted ratio comparison:")
    print(sorted_ablation_df.to_string(index=False))
    print("\n" + "-"*50 + "\n")

    # Programmatically select the best historical training ratio parameter safely
    selected_ratio = sorted_ablation_df.loc[0, 'Seen Ratio']
    selected_unseen = sorted_ablation_df.loc[0, 'Unseen Ratio']

    print("Automatically selected host split ratio:")
    print(f"Seen/Unseen = {selected_ratio}/{selected_unseen}")

    # Sync structural table straight onto your Drive target results workspace path
    csv_out_path = os.path.join(CONFIG['results_dir'], 'host_ratio_ablation_study.csv')
    ablation_df.to_csv(csv_out_path, index=False)
    print(f"Saved: {csv_out_path}\n")
    print("=" * 115)
    print("                                      FINAL PARALYZED SPLIT SUMMARY                                  ")
    print("=" * 115)

    # Extract selected optimal data partitions from memory cache
    t_df, v_df, h_df, tm_df, hd_df = partition_caches[selected_ratio]

    # Print inline population breakdowns matching requested pandas visual series text structures
    print(f"\nTrain: rows={len(t_df):,}, hosts={t_df['hostid_str'].nunique()}, labels={t_df['label'].nunique()}")
    print(t_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nValidation: rows={len(v_df):,}, hosts={v_df['hostid_str'].nunique()}, labels={v_df['label'].nunique()}")
    print(v_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nHost-Test: rows={len(h_df):,}, hosts={h_df['hostid_str'].nunique()}, labels={h_df['label'].nunique()}")
    print(h_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nTime-Test: rows={len(tm_df):,}, hosts={tm_df['hostid_str'].nunique()}, labels={tm_df['label'].nunique()}")
    print(tm_df['label'].value_counts().to_string())
    print("\n" + "-"*40)

    print(f"\nHard-Test: rows={len(hd_df):,}, hosts={hd_df['hostid_str'].nunique()}, labels={hd_df['label'].nunique()}")
    print(hd_df['label'].value_counts().to_string())
    print("=" * 115)

    return selected_ratio

# Execute updated study directly inside your execution cell block sequence
best_seen_ratio = run_host_ablation_study(df_clean, expected_features)

In [ ]:
# BLOCK 6: STAGE 8 — PRE-REQUISITE CHECKPOINT: ZERO-DAY HOLDOUT QUARANTINE CORE (FIXED)
print(f"[Quarantine Layer] Isolating {CONFIG['zero_day_class']} from model ingestion parameters...")

# STRUCTURAL FIX: Re-initialize the position-invariant host feature mapping inside this split scope
df_clean = df_clean.copy()
df_clean['hostid'] = df_clean['src_ip'].astype(str).map(lambda x: x.split('.')[-1] if '.' in x else '1')

# Programmatically align with the dynamically chosen ablation study ratio from Stage 2
try:
    target_ratio = best_seen_ratio
except NameError:
    # Safe structural fallback to 0.6 if block memory spaces were completely reset
    target_ratio = 0.6

# Isolate the zero-day class entirely to prevent validation data leakage
df_train_pool = df_clean[df_clean['label'] != CONFIG['zero_day_class']].copy()
df_zero_day_isolated = df_clean[df_clean['label'] == CONFIG['zero_day_class']].copy()

unique_hosts = df_train_pool['hostid'].unique()
np.random.seed(RANDOM_STATE)
seen_hosts = np.random.choice(unique_hosts, size=int(len(unique_hosts) * target_ratio), replace=False)

# Tag hosts as 'Seen' or 'Unseen' to create topological evaluation boundaries
df_train_pool['host_group'] = df_train_pool['hostid'].map(lambda x: 'Seen' if x in seen_hosts else 'Unseen')

# Stratify samples cross-sectionally to populate the five deployment vectors
train_df = df_train_pool[df_train_pool['host_group'] == 'Seen'].sample(frac=0.6, random_state=RANDOM_STATE)
val_df = df_train_pool[df_train_pool['host_group'] == 'Unseen'].sample(frac=0.2, random_state=RANDOM_STATE)
host_test_df = df_train_pool[df_train_pool['host_group'] == 'Unseen'].sample(frac=0.2, random_state=RANDOM_STATE)
time_test_df = df_train_pool[df_train_pool['host_group'] == 'Seen'].sample(frac=0.2, random_state=RANDOM_STATE)
hard_test_df = df_train_pool[df_train_pool['host_group'] == 'Unseen'].sample(frac=0.2, random_state=RANDOM_STATE)

# Convert structured data splits into position-invariant numpy arrays for LightGBM
X_train = train_df[expected_features].to_numpy()
y_train = train_df['label'].map(GLOBAL_CLASS_INDEX).to_numpy()
X_val = val_df[expected_features].to_numpy()
y_val = val_df['label'].map(GLOBAL_CLASS_INDEX).to_numpy()

print("\n" + "="*80)
print("             QUARANTINE BLOCK SEPARATION COMPLETED SUCCESSFULLY              ")
print("="*80)
print(f"[*] Training Target Matrix Dimensions:     {X_train.shape}")
print(f"[*] Validation Target Matrix Dimensions:   {X_val.shape}")
print(f"[*] Quarantined Zero-Day Sample Points:    {len(df_zero_day_isolated):,}")
print(f"[*] Data Leakage Verification Check:        Is zero-day target inside y_train? -> {GLOBAL_CLASS_INDEX[CONFIG['zero_day_class']] in y_train}")
print("=" * 80)

In [ ]:
# BLOCK 7: STAGE 3 — PRIMARY MODEL SEEDING AND PROBABILITY TUNING
print("[Stage 3] Training invariant tree optimization layer paths...")
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'objective': 'multiclass',
    'num_class': 12,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 32,
    'random_state': RANDOM_STATE,
    'verbose': -1
}

lgb_model = lgb.train(
    params, train_data, num_boost_round=100,
    valid_sets=[val_data], callbacks=[lgb.early_stopping(10, verbose=False)]
)

print("[Calibration Layer] Aligning raw output thresholds with Isotonic Regression...")
val_raw_preds = lgb_model.predict(X_val)
calibrators = {}

for c in range(12):
    ir = IsotonicRegression(out_of_bounds='clip')
    target_binary = (y_val == c).astype(int)
    if len(np.unique(target_binary)) < 2:
        target_binary = np.append(target_binary, [0, 1])
        val_raw_preds_padded = np.append(val_raw_preds[:, c], [0.0, 1.0])
        ir.fit(val_raw_preds_padded, target_binary)
    else:
        ir.fit(val_raw_preds[:, c], target_binary)
    calibrators[c] = ir

def get_calibrated_probabilities(model, idx_calibrators, X_mat):
    raw = model.predict(X_mat)
    calibrated = np.zeros_like(raw)
    for cls_idx in range(12):
        calibrated[:, cls_idx] = idx_calibrators[cls_idx].transform(raw[:, cls_idx])
    sums = calibrated.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1.0
    return calibrated / sums

print("[Stage 3 Complete] Probability channels normalized successfully.")

In [ ]:
# BLOCK 8: STAGE 4 — SHAP BEHAVIORAL ATTRIBUTION PROFILE ENGINE SIGNATURES (SHAPE CORRECTION FIXED)
def compute_shap_signatures(model, X_ref, df_meta, feature_names):
    print("[Stage 4] Compiling continuous game-theoretic feature consensus metrics...")
    explainer = shap.TreeExplainer(model)

    # Structural density group sampler matching historical sample parameters
    sample_rows = df_meta.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 40), random_state=RANDOM_STATE)
    ).reset_index(drop=True)

    X_sample_numpy = sample_rows[feature_names].to_numpy()
    shap_values = explainer.shap_values(X_sample_numpy)

    signatures = {}
    variance_audits = {}

    for class_name, class_idx in GLOBAL_CLASS_INDEX.items():
        mask = (sample_rows['label'] == class_name).to_numpy()
        if not mask.any(): continue

        # FIXED: Dynamic shape detection handler to insulate against 3D tensor vs List slicing variations
        if isinstance(shap_values, list):
            # Traditional list of arrays format: isolate class list element first, then slice samples
            class_shap = shap_values[class_idx][mask]
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            # Modern 3D NumPy tensor: shap_values shape is (samples, features, classes)
            # Filter samples along axis 0 matching the mask, then isolate the target class index
            class_shap = shap_values[mask, :, class_idx]
        else:
            # Emergency fallback structure encapsulation
            try:
                class_shap = np.array(shap_values)[class_idx][mask]
            except Exception:
                class_shap = shap_values[mask]

        # Row-wise L2 spatial normalization layer
        norms = np.linalg.norm(class_shap, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        normalized_shap = class_shap / norms

        # Extract mean direction consensus blueprint vector
        mean_vector = normalized_shap.mean(axis=0)
        vec_norm = np.linalg.norm(mean_vector)
        signatures[class_name] = mean_vector / (vec_norm if vec_norm > 0 else 1.0)
        variance_audits[class_name] = np.sqrt(normalized_shap.var(axis=0))

    return signatures, variance_audits

# Execute baseline calculations directly inline without wall-of-text print statements
train_signatures, train_variances = compute_shap_signatures(lgb_model, X_train, train_df, expected_features)
print(f"[Stage 4 Complete] Unique Signature vectors generated count: {len(train_signatures)}")

In [ ]:
# BLOCK 9: STAGE 5 — AUTHENTIC SHAP VS LIME CORRELATION INTEGRITY AGREEMENT CHECK

!pip install lime --quiet
from lime.lime_tabular import LimeTabularExplainer

def run_explanation_cross_validation(model, X_matrix, feature_names, df_train_reference):
    print("[Stage 5] Launching True SHAP vs LIME Explanation Cross-Validation Engine...")

    # 1. Initialize True LIME Tabular Explainer over training baseline characteristics
    lime_explainer = LimeTabularExplainer(
        training_data=X_matrix,
        feature_names=feature_names,
        class_names=list(GLOBAL_CLASS_INDEX.keys()),
        mode='classification',
        random_state=RANDOM_STATE
    )

    # 2. Initialize SHAP Explainer
    shap_explainer = shap.TreeExplainer(model)

    sample_size = min(20, len(X_matrix)) # Core sample subset for execution efficiency
    raw_shap_values = shap_explainer.shap_values(X_matrix[:sample_size])

    raw_preds = model.predict(X_matrix[:sample_size])
    predicted_classes = np.argmax(raw_preds, axis=1)

    jaccard_scores = []

    for idx in range(sample_size):
        target_class_idx = predicted_classes[idx]

        # --- A. Isolate True SHAP Top Features ---
        if isinstance(raw_shap_values, np.ndarray) and raw_shap_values.ndim == 3:
            shap_attr = raw_shap_values[idx, :, target_class_idx]
        elif isinstance(raw_shap_values, list):
            shap_attr = raw_shap_values[target_class_idx][idx]
        else:
            shap_attr = raw_shap_values[idx]
        top_shap_features = set(np.argsort(np.abs(shap_attr))[-5:])

        # --- B. Generate True LIME Top Features ---
        # Define inline prediction delegate for LIME's local linear model
        predict_fn = lambda x: get_calibrated_probabilities(model, calibrators, x)

        exp = lime_explainer.explain_instance(
            data_row=X_matrix[idx],
            predict_fn=predict_fn,
            num_features=len(feature_names),
            labels=(target_class_idx,)
        )

        # Map LIME's text/index outputs back into pure feature column positions
        local_exp_list = exp.as_map()[target_class_idx]
        top_lime_features = set([feat_idx for feat_idx, weight in sorted(local_exp_list, key=lambda x: abs(x[1]))[-5:]])

        # --- C. Compute Jaccard Intersection Over Union ---
        intersection = len(top_shap_features & top_lime_features)
        union = len(top_shap_features | top_lime_features)
        jaccard_scores.append(intersection / union if union > 0 else 0.0)

    mean_jaccard = np.mean(jaccard_scores)
    print(f"[Stage 5 Metric Verification] Authentic Mean Top-5 Jaccard Agreement: {mean_jaccard:.4f}")

    # =====================================================================
    # VISUALIZATION GENERATION LAYER
    # =====================================================================
    plt.figure(figsize=(12, 6.5))
    sample_indices = [f"Sample {i+1}" for i in range(sample_size)]

    bars = plt.barh(sample_indices, jaccard_scores, color='steelblue', edgecolor='black', alpha=0.9, height=0.6)
    plt.axvline(mean_jaccard, color='crimson', linestyle='--', linewidth=2,
                label=f'Mean Agreement Bound ({mean_jaccard:.4f})')

    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.2f}',
                 va='center', ha='left', fontsize=9, fontweight='semibold', color='black')

    plt.xlabel('Top-5 Feature Jaccard Intersection Agreement Index', fontsize=11, fontweight='bold', labelpad=10)
    plt.ylabel('Evaluation Network Flow Index Slices', fontsize=11, fontweight='bold', labelpad=10)
    plt.title('Agnostic Attribution Space Integrity Consensus (True SHAP vs True Tabular LIME Explanations)',
              fontsize=12, fontweight='bold', pad=15)

    plt.xlim(0, 1.15)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='gray')
    plt.tight_layout()

    local_fig_path = os.path.join(RESULTS_DIR, 'figures', 'shap_lime_jaccard_agreement.png')
    plt.savefig(local_fig_path, dpi=300)
    try:
        drive_fig_path = os.path.join(CONFIG['results_dir'], 'shap_lime_jaccard_agreement.png')
        plt.savefig(drive_fig_path, dpi=300)
        print(f"[Visualization Saved] Plot synced to Drive: {drive_fig_path}")
    except Exception:
        print(f"[Visualization Saved] Plot written to local workspace: {local_fig_path}")

    plt.show()
    return mean_jaccard

# Pass variables directly inside your block pipeline cell execution sequence
global_xai_agreement = run_explanation_cross_validation(lgb_model, X_val, expected_features, train_df)

In [ ]:
# BLOCK 10: STAGE 6 — AUTOMATED POLYNOMIAL GRID COORDINATOR (STRICT VALIDATION COST EXTRACTION)

import seaborn as sns


def run_automated_grid_search_engine(model, idx_calibrators, reference_signatures, X_val_mat, y_val_mat):
    print("=" * 115)
    print("[Automated Grid Search] Initiating Systematic Hyperparameter Optimization Suite...")
    print("=" * 115)

    # 1. Extract base model probability space and game-theoretic attributions once to minimize latency
    probs = get_calibrated_probabilities(model, idx_calibrators, X_val_mat)
    preds = np.argmax(probs, axis=1)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val_mat)

    # 2. Define a granular, high-density coordinate grid space
    tp_grid = np.linspace(0.60, 0.98, 20)  # 20 steps of confidence thresholds
    ts_grid = np.linspace(0.01, 0.40, 20)  # 20 steps of structural similarity thresholds

    # Pre-allocate evaluation tracking dataframes for our surface map
    search_results_log = []

    # Initialize our safe baseline fallback coordinates
    best_tp, best_ts = 0.95, 0.20
    lowest_operational_cost = float('inf')

    # 3. Begin systematic coordinate sweep
    for tp_cand in tp_grid:
        for ts_cand in ts_grid:
            missed, false_blocks, escalated, total_malicious = 0, 0, 0, 0

            for i in range(len(X_val_mat)):
                p_idx = preds[i]
                p_name = REVERSE_CLASS_INDEX[p_idx]
                t_name = REVERSE_CLASS_INDEX[y_val_mat[i]]

                attack_confidence = 1.0 - probs[i, GLOBAL_CLASS_INDEX['Benign']]
                if t_name != 'Benign': total_malicious += 1

                if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                    inst_shap = shap_values[i, :, p_idx]
                elif isinstance(shap_values, list):
                    inst_shap = shap_values[p_idx][i]
                else:
                    inst_shap = shap_values[i]

                norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)
                similarity = np.dot(norm_inst, reference_signatures.get(p_name, np.zeros_like(norm_inst)))

                # --- Multi-Tier Policy Gate Calculus Routing ---
                if p_name == 'Benign' and attack_confidence < tp_cand and similarity >= ts_cand:
                    if t_name != 'Benign': missed += 1
                elif p_name != 'Benign' and attack_confidence >= tp_cand and similarity >= ts_cand:
                    if t_name == 'Benign': false_blocks += 1
                elif p_name != 'Benign' and attack_confidence >= tp_cand and similarity >= (ts_cand * 0.5):
                    if t_name == 'Benign': false_blocks += 1
                elif p_name == 'Benign' and attack_confidence >= tp_cand and similarity < ts_cand:
                    if t_name == 'Benign': false_blocks += 1
                else:
                    if p_name == 'Benign' and t_name != 'Benign':
                        missed += 1
                    else:
                        escalated += 1

            anmr = (total_malicious - missed) / (total_malicious + 1e-9)
            fbr = false_blocks / (len(X_val_mat) - total_malicious + 1e-9)
            esc_rate = escalated / len(X_val_mat)

            # --- CONSOLIDATED OPERATIONAL OBJECTIVE COST FUNCTION ---
            # W1 (10.0): Security Breach Penalty (Massive penalty if attacks slip past)
            # W2 (15.0): False Positive Perimeter Disruption Penalty
            # W3 (8.0):  Analyst Alert Fatigue / Escalation Queue Penalty
            operational_cost = ((1.0 - anmr) * 10.0) + (fbr * 15.0) + (esc_rate * 8.0)

            # Record telemetry for surface plotting
            search_results_log.append({
                'Tau_P': tp_cand,
                'Tau_S': ts_cand,
                'ANMR': anmr,
                'FBR': fbr,
                'Escalation_Rate': esc_rate,
                'Operational_Cost': operational_cost
            })

            # Strict security constraint enforcement: ANMR must meet minimal protection guidelines
            if operational_cost < lowest_operational_cost and anmr >= 0.75:
                lowest_operational_cost = operational_cost
                best_tp = tp_cand
                best_ts = ts_cand

    print(f"\n[Optimization Complete] Global Absolute Minimum Isolated!")
    print(f" -> Optimal Confident Boundary (Tau_P): {best_tp:.2f}")
    print(f" -> Optimal Structural Boundary (Tau_S): {best_ts:.2f}")
    print(f" -> Minimum Validation Operational Cost: {lowest_operational_cost:.4f}")

    # 4. Generate the empirical validation figures for the manuscript
    df_surf = pd.DataFrame(search_results_log)
    pivot_surf = df_surf.pivot(index='Tau_P', columns='Tau_S', values='Operational_Cost')

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_surf, cmap='viridis_r', cbar_kws={'label': 'Operational Cost Metric Value'})
    plt.title('Empirical Perimeter Optimization Space:\nValidation Dataset Cost Minimization Topology', fontsize=12, fontweight='bold')
    plt.xlabel('Structural Similarity Threshold ($\\tau_s$)', fontsize=10)
    plt.ylabel('Calibrated Prediction Confidence Threshold ($\\tau_p$)', fontsize=10)

    # Highlight the absolute mathematical minimum choice with an indicator mark
    plt.tight_layout()
    plt.show()

    return best_tp, best_ts

# Execute authentic, un-leaked tuning over validation data structures exclusively
TAU_P, TAU_S = run_automated_grid_search_engine(lgb_model, calibrators, train_signatures, X_val, y_val)

In [ ]:
# BLOCK 11: SYSTEM OPERATIONAL LOGGING ENGINE (AUTONOMOUS FORCE-BLOCK LAYER)
def evaluate_operational_environment(model, idx_calibrators, reference_signatures, tp, ts, X_matrix, y_matrix, environment_tag):
    start_time = time.time()
    proc = psutil.Process(os.getpid())
    memory_baseline = proc.memory_info().rss / (1024 * 1024)

    probs = get_calibrated_probabilities(model, idx_calibrators, X_matrix)
    preds = np.argmax(probs, axis=1)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_matrix)

    unique_classes_present = len(np.unique(y_matrix))

    missed, false_blocks, escalated, total_malicious = 0, 0, 0, 0
    for i in range(len(X_matrix)):
        p_idx = preds[i]
        p_name = REVERSE_CLASS_INDEX[p_idx]
        t_name = REVERSE_CLASS_INDEX[y_matrix[i]]

        attack_confidence = 1.0 - probs[i, GLOBAL_CLASS_INDEX['Benign']]
        if t_name != 'Benign': total_malicious += 1

        # --- EXTRACT RUNTIME XAI ATTRIBUTION VECTOR ---
        if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            inst_shap = shap_values[i, :, p_idx]
        elif isinstance(shap_values, list):
            inst_shap = shap_values[p_idx][i]
        else:
            inst_shap = shap_values[i]

        norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)

        # Calculate alignment against baseline blueprints
        if p_name in reference_signatures:
            similarity = np.dot(norm_inst, reference_signatures[p_name])
        else:
            similarity = 0.0

        # =====================================================================
        # ADVANCED TIERED SECURITY ROUTING POLICY (AUTONOMOUS BLOCKS OPTIMIZED)
        # =====================================================================
        # Tier 1: Confident, Verified Benign Pass
        if p_name == 'Benign' and attack_confidence < tp and similarity >= ts:
            if t_name != 'Benign': missed += 1

        # Tier 2A: Perfect Canonical Autonomous Block
        elif p_name != 'Benign' and attack_confidence >= tp and similarity >= ts:
            if t_name == 'Benign': false_blocks += 1

        # Tier 2B: Defensive Topological Drift Mitigation Block
        elif p_name != 'Benign' and attack_confidence >= tp and similarity >= (ts * 0.5):
            if t_name == 'Benign': false_blocks += 1

        # Tier 2C: NEW ENGINE ADDITION — AUTONOMOUS FORCE-BLOCK OF DRIFTED TRAFFIC
        # If the model says Benign, but the backup mathematical risk is severe AND the
        # explanation shape diverges from normal benign behavior, drop the escalation and BLOCK it.
        elif p_name == 'Benign' and attack_confidence >= tp and similarity < ts:
            # Shifted straight into a block action!
            if t_name == 'Benign': false_blocks += 1

        # Tier 3: True Structural Chaos / Pure Ambiguity Anomaly Queue
        else:
            if p_name == 'Benign' and t_name != 'Benign':
                missed += 1  # Borderline drift slip
            else:
                escalated += 1

    execution_latency = (time.time() - start_time) / len(X_matrix)
    memory_terminal = proc.memory_info().rss / (1024 * 1024)

    anmr_final = (total_malicious - missed) / (total_malicious + 1e-9)
    fbr_final = false_blocks / (len(X_matrix) - total_malicious + 1e-9)

    return {
        'Target_Environment': environment_tag,
        'Label_Diversity_Score': unique_classes_present,
        'Global_Accuracy': accuracy_score(y_matrix, preds),
        'Macro_F1_Throughput': f1_score(y_matrix, preds, average='macro', zero_division=0),
        'Attack_Not_Missed_Rate_ANMR': anmr_final,
        'False_Block_Rate_FBR': fbr_final,
        'Escalation_Rate': escalated / len(X_matrix),
        'Latency_ms_per_packet': execution_latency * 1000,
        'Throughput_packets_per_sec': 1.0 / (execution_latency + 1e-9),
        'RAM_Footprint_Overhead_MB': max(0.0, memory_terminal - memory_baseline)
    }

print("[System Core Engine] Autonomous Block Optimization Policy compiled successfully.")

In [ ]:
# BLOCK 12: STAGE 7 — STRESS-TESTS ACROSS NATURAL COVARIATE DRIFT MATRICES (POLICY OVERRIDE RUN)
print("=" * 115)
print("[Stage 7] Actively tracking metrics across Natural Concept Drift partitions (Tiered Override Enabled)...")
print("=" * 115)

drift_environments_list = [
    ('Host-Test (Topological Shift)', host_test_df),
    ('Time-Test (Temporal Drift)', time_test_df),
    ('Hard-Test (Combined Shift)', hard_test_df)
]

if 'comprehensive_metrics_log' not in locals() and 'comprehensive_metrics_log' not in globals():
    comprehensive_metrics_log = []

block_12_records = []

# FORCE COMPILATION FIXED LOOP DIRECTLY IN BLOCK 12 SCOPE
for tag, df_slice in drift_environments_list:
    X_test_env = df_slice[expected_features].to_numpy()
    y_test_env = df_slice['label'].map(GLOBAL_CLASS_INDEX).to_numpy()

    # Run using your optimized parameters (TAU_P=0.95, TAU_S=0.2)
    # The Block 11 update will catch the data points and process them through the Tiered buffer
    metrics = evaluate_operational_environment(lgb_model, calibrators, train_signatures, TAU_P, TAU_S, X_test_env, y_test_env, tag)

    comprehensive_metrics_log.append(metrics)
    block_12_records.append(metrics)

# --- INSTANT DISPLAY FORMATTING ENGINE ---
block_12_df = pd.DataFrame(block_12_records)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n" + "#" * 115)
print("             STAGE 7: RE-EVALUATED TIERED SAFETY GATE RESULTS           ")
print("#" * 115)
print(block_12_df.to_string(index=False, formatters={
    'Global_Accuracy': '{:,.4f}'.format,
    'Macro_F1_Throughput': '{:,.4f}'.format,
    'Attack_Not_Missed_Rate_ANMR': '{:,.4f}'.format,
    'False_Block_Rate_FBR': '{:,.4f}'.format,
    'Escalation_Rate': '{:,.4f}'.format,
    'Latency_ms_per_packet': '{:,.3f} ms'.format,
    'Throughput_packets_per_sec': '{:,.1f} pps'.format,
    'RAM_Footprint_Overhead_MB': '{:,.2f} MB'.format
}))
print("#" * 115 + "\n")

In [ ]:
# DIAGNOSTIC SUB-CELL: ENTIRE DRIFT VECTOR PIECE-WISE POLICY MATRIX AUDIT


print("=" * 115)
print("             COMPLETE NATURAL CONCEPT DRIFT CORE SECURITY & POLICY ROUTING AUDIT            ")
print("=" * 115)

# Build the complete evaluation collection matrix
all_drift_environments = [
    ('Host-Test (Topological Shift Only)', host_test_df),
    ('Time-Test (Temporal Drift Only)', time_test_df),
    ('Hard-Test (Combined Spatial-Temporal Shift)', hard_test_df)
]

explainer_diag = shap.TreeExplainer(lgb_model)
compiled_report_data = []

for partition_tag, df_slice in all_drift_environments:
    print(f"[Auditing Engine] Generating full explanation mapping arrays for: {partition_tag}...")

    X_diag = df_slice[expected_features].to_numpy()
    y_diag = df_slice['label'].map(GLOBAL_CLASS_INDEX).to_numpy()

    probs_diag = get_calibrated_probabilities(lgb_model, calibrators, X_diag)
    preds_diag = np.argmax(probs_diag, axis=1)
    shap_values_diag = explainer_diag.shap_values(X_diag)

    # Reset metrics telemetry channels for the current partition scope
    tier_1_benign_pass = 0
    tier_2a_canonical_block = 0
    tier_2b_drift_mitigation_block = 0
    tier_3_administrative_escalation = 0
    total_malicious_flows = 0
    missed_attacks = 0

    for i in range(len(X_diag)):
        p_idx = preds_diag[i]
        p_name = REVERSE_CLASS_INDEX[p_idx]
        t_name = REVERSE_CLASS_INDEX[y_diag[i]]

        if t_name != 'Benign':
            total_malicious_flows += 1

        attack_confidence = 1.0 - probs_diag[i, GLOBAL_CLASS_INDEX['Benign']]

        # --- EXTRACT RUNTIME XAI ATTRIBUTION VECTOR ---
        if isinstance(shap_values_diag, np.ndarray) and shap_values_diag.ndim == 3:
            inst_shap = shap_values_diag[i, :, p_idx]
        elif isinstance(shap_values_diag, list):
            inst_shap = shap_values_diag[p_idx][i]
        else:
            inst_shap = shap_values_diag[i]

        norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)

        # Determine geometric alignment against historical training baseline signature blueprints
        if p_name in train_signatures:
            similarity = np.dot(norm_inst, train_signatures[p_name])
        else:
            similarity = 0.0

        # --- MACHINE LEARNING POLICY GATE ROUTING CALCULUS ---
        if p_name == 'Benign' and attack_confidence < TAU_P:
            # Tier 1: Evaluated and cleared as standard Benign data stream traffic
            tier_1_benign_pass += 1
            if t_name != 'Benign':
                missed_attacks += 1

        elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity >= TAU_S:
            # Tier 2A: Confident attack classification supported by canonical SHAP signatures
            tier_2a_canonical_block += 1

        elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity >= (TAU_S * 0.5):
            # Tier 2B: Confident attack classification supported by partial signature cushion (Topological Shift Buffer)
            tier_2b_drift_mitigation_block += 1

        else:
            # Tier 3: Divergent, unaligned, or borderline anomalies dropped safely into escalation
            tier_3_administrative_escalation += 1

    # Record metrics into structural rows for summary layout creation
    compiled_report_data.append({
        'Environment Partition': partition_tag,
        'Total Traffic Packets': len(X_diag),
        'Total Attacks Present': total_malicious_flows,
        'Tier 1 (Benign Pass)': tier_1_benign_pass,
        'Tier 2A (Canonical Block)': tier_2a_canonical_block,
        'Tier 2B (Drift Block)': tier_2b_drift_mitigation_block,
        'Tier 3 (Escalated)': tier_3_administrative_escalation,
        'Silently Missed (Breaches)': missed_attacks
    })

# --- RENDER MASTER TELEMETRY TABLE MATRIX ---
master_audit_df = pd.DataFrame(compiled_report_data)
print("\n" + "#" * 115)
print("                    CRITICAL PERIMETER GATE PERFORMANCE SUB-CELL LOGS (ALL DRIFT SHIFTS)                   ")
print("#" * 115)
print(master_audit_df.to_string(index=False))
print("#" * 115 + "\n")

In [ ]:
# BLOCK 13: STAGE 8 — CONCEPT EVOLUTION: WEB_BRUTE_FORCE ZERO-DAY EMERGENCY HOLDOUT (WITH LIVE PRINT)
print("=" * 115)
print(f"[Stage 8] Evaluating un-encountered threat vectors for zero-day holdout: {CONFIG['zero_day_class']}...")
print("=" * 115)

if 'df_zero_day_isolated' in locals() or 'df_zero_day_isolated' in globals():
    if len(df_zero_day_isolated) > 0:
        X_zday = df_zero_day_isolated[expected_features].to_numpy()
        y_zday = df_zero_day_isolated['label'].map(GLOBAL_CLASS_INDEX).to_numpy()

        # Invoke the shape-corrected evaluator from Block 11
        zday_metrics = evaluate_operational_environment(
            lgb_model, calibrators, train_signatures, TAU_P, TAU_S, X_zday, y_zday,
            f"Stage 8: Zero-Day Concept Evolution ({CONFIG['zero_day_class']} Holdout)"
        )

        # Append quietly for downstream Block 16 consolidation
        comprehensive_metrics_log.append(zday_metrics)

        # --- INSTANT CELL TERMINAL DISPLAY FORMATTING ENGINE ---
        block_13_df = pd.DataFrame([zday_metrics])

        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)

        print("\n" + "#" * 115)
        print(f"             STAGE 8: CONCEPT EVOLUTION ZERO-DAY HOLDOUT EVALUATION RESULTS ({CONFIG['zero_day_class']})           ")
        print("#" * 115)
        print(block_13_df.to_string(index=False, formatters={
            'Global_Accuracy': '{:,.4f}'.format,
            'Macro_F1_Throughput': '{:,.4f}'.format,
            'Attack_Not_Missed_Rate_ANMR': '{:,.4f}'.format,
            'False_Block_Rate_FBR': '{:,.4f}'.format,
            'Escalation_Rate': '{:,.4f}'.format,
            'Latency_ms_per_packet': '{:,.3f} ms'.format,
            'Throughput_packets_per_sec': '{:,.1f} pps'.format,
            'RAM_Footprint_Overhead_MB': '{:,.2f} MB'.format
        }))
        print("#" * 115 + "\n")
        print(f"[Stage 8 Verification] Unknown Threat Escalation Capture Efficiency: {zday_metrics['Escalation_Rate']:.4f}")
    else:
        print("[Stage 8 Warning] Isolate target dataframe rows are empty. Verify preprocessing distributions.")
else:
    print("[Stage 8 Error] Isolated zero-day partition matrices not initialized in memory space. Re-run Block 6.")

In [ ]:
# DIAGNOSTIC SUB-CELL: AUDITING THE CHANNELS OF THE 18.11% ESCALATION RATE
print("=" * 90)
print("             ZERO-DAY ESCALATION QUEUE PROFILE: SCENARIO DRIFT AUDIT             ")
print("=" * 90)

# Ensure the zero-day matrix is available
if 'df_zero_day_isolated' in locals() and len(df_zero_day_isolated) > 0:
    X_zday_diag = df_zero_day_isolated[expected_features].to_numpy()

    # 1. Compute calibrated probabilities and model predictions
    probs_diag = get_calibrated_probabilities(lgb_model, calibrators, X_zday_diag)
    preds_diag = np.argmax(probs_diag, axis=1)

    # 2. Extract local SHAP explanations for the zero-day set
    explainer_diag = shap.TreeExplainer(lgb_model)
    shap_values_diag = explainer_diag.shap_values(X_zday_diag)

    # 3. Initialize counter metrics
    total_packets = len(X_zday_diag)
    autonomous_blocks = 0
    scenario_a_count = 0  # Low-confidence Benign predictions
    scenario_b_count = 0  # High-confidence attack with broken attribution shapes
    other_escalations = 0

    # Dictionary to keep track of what known classes the blind model guessed
    misclassification_tracker = defaultdict(int)

    for i in range(total_packets):
        p_idx = preds_diag[i]
        p_name = REVERSE_CLASS_INDEX[p_idx]
        misclassification_tracker[p_name] += 1

        # Calculate overall attack risk
        attack_confidence = 1.0 - probs_diag[i, GLOBAL_CLASS_INDEX['Benign']]

        # Calculate explanation similarity against baseline signatures
        if p_name in train_signatures:
            if isinstance(shap_values_diag, np.ndarray) and shap_values_diag.ndim == 3:
                inst_shap = shap_values_diag[i, :, p_idx]
            elif isinstance(shap_values_diag, list):
                inst_shap = shap_values_diag[p_idx][i]
            else:
                inst_shap = shap_values_diag[i]

            norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)
            similarity = np.dot(norm_inst, train_signatures[p_name])
        else:
            similarity = 0.0

        # --- ROUTING ANALYSIS CRITERIA ---
        if p_name == 'Benign' and attack_confidence < TAU_P:
            # Safely passed as benign (Should be 0 since this is an actual attack)
            pass
        elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity >= TAU_S:
            # Path 1: Autonomous Block (recognizing shared traits with known attacks)
            autonomous_blocks += 1
        else:
            # Path 2: Escalated (The 18.11% footprint)
            if p_name == 'Benign' and attack_confidence >= TAU_P:
                # Scenario A: Model guessed Benign, but the backup attack risk is too high
                scenario_a_count += 1
            elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity < TAU_S:
                # Scenario B: Model guessed an attack, but the SHAP explanation shape is broken
                scenario_b_count += 1
            else:
                # Catch-all for borderline low-confidence attack predictions (< TAU_P)
                other_escalations += 1

    # --- PRINT OUT ANALYSIS TABLES ---
    print(f"Total Zero-Day (Web_Brute_Force) Packets Evaluated: {total_packets:,}")
    print(f"Natively Blocked by Safety Gate (Autonomous Actions): {autonomous_blocks:,} ({autonomous_blocks/total_packets*100:.2f}%)")
    print(f"Total Escalated Packets (Human Validation Queue):    {scenario_a_count + scenario_b_count + other_escalations:,} ({(scenario_a_count + scenario_b_count + other_escalations)/total_packets*100:.2f}%)")

    print("\n" + "-"*50)
    print("ESCALATION BACKPLANE BREAKDOWN:")
    print("-"*50)
    print(f"[*] Scenario A (Low-Confidence Benign):           {scenario_a_count:,} packets ({(scenario_a_count/total_packets)*100:.2f}%)")
    print(f"    -> Meaning: Model predicted 'Benign', but calibrated risk crossed Tau_P ({TAU_P}).")
    print(f"[*] Scenario B (Structural Shape Anomaly):        {scenario_b_count:,} packets ({(scenario_b_count/total_packets)*100:.2f}%)")
    print(f"    -> Meaning: Model predicted an attack with high confidence, but feature attribution similarity fell below Tau_S ({TAU_S}).")
    print(f"[*] Other Boundary Hesitations:                   {other_escalations:,} packets ({(other_escalations/total_packets)*100:.2f}%)")
    print(f"    -> Meaning: Model couldn't comfortably cross prediction limits for any class.")

    print("\n" + "-"*50)
    print("HOW THE BLIND CLASSIFIER MISCLASSIFIED THE ZERO-DAY:")
    print("-"*50)
    for target_label, count_vals in sorted(misclassification_tracker.items(), key=lambda x: x[1], reverse=True):
        print(f" -> Predicted as class [{target_label:<16}]: {count_vals:,} times ({(count_vals/total_packets)*100:.2f}%)")
    print("=" * 90)
else:
    print("[Diagnostic Error] Zero-day array isolate cache missing from runtime storage memory.")

In [ ]:
# BLOCK 14: STAGE 9 — REAL-WORLD SCHEMA-ALIGNED TRUE CROSS-DATASET TRANSFER RESULTS
import os
import glob
import numpy as np
import pandas as pd

print("=" * 115)
print("[Stage 9] Launching Real-World Schema-Aligned Cross-Dataset True External Transfers...")
print("=" * 115)

external_targets_pool = ["CIC-IDS2017", "CSE-CIC-IDS2018"]
block_14_records = []

# --- EXPLICIT DIAGNOSTIC INTERCEPTION REGISTER ---
globals()['STAGE_9_CAPTURED_MATRICES'] = {}

def aggregate_and_stratify_drive_directory(target_dataset_tag, total_target_samples=1000):
    """
    Dynamically logs into Google Drive folders, reads available CSV assets,
    maps raw label arrays to internal classes, and forces a balanced, stratified
    sample cohort across all identifiable threat signatures while strictly skipping Zero-Day holdouts.
    """
    dir_key = 'cic2017_dir' if target_dataset_tag == "CIC-IDS2017" else 'cse2018_dir'
    target_dir = CONFIG[dir_key]

    if not os.path.exists(target_dir):
        raise FileNotFoundError(f"Google Drive directory target completely missing at: {target_dir}")

    csv_files = glob.glob(os.path.join(target_dir, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No valid raw dataset files (.csv) isolated inside: {target_dir}")

    collected_chunks = []
    print(f" -> Scanning files inside folder path...")

    for file_path in csv_files:
        try:
            chunk_df = pd.read_csv(file_path, low_memory=False)
            if not chunk_df.empty:
                # Standardize column headers instantly to find the label column securely
                chunk_df.columns = [str(c).strip() for c in chunk_df.columns]
                collected_chunks.append(chunk_df)
        except Exception as e:
            print(f"[Loader Warning] Bypassing asset {os.path.basename(file_path)} due to: {str(e)}")

    if not collected_chunks:
        raise ValueError(f"Failed to gather clean data lines from isolated assets in {target_dir}")

    aggregated_df = pd.concat(collected_chunks, ignore_index=True)

    # Locate the target raw label column dynamically
    label_col_marker = [c for c in aggregated_df.columns if c.lower() == 'label'][0]

    # Advanced Clean: Replace non-ascii artifacts (like '') common in raw Web Attack labels
    raw_labels_cleaned = aggregated_df[label_col_marker].astype(str).str.strip().str.replace(r'[^\x00-\x7F]+', '_', regex=True)

    # Map raw string labels to standardized internal target classifications
    aggregated_df['standardized_target_label'] = raw_labels_cleaned.map(
        lambda x: CANONICAL_LABEL_MAP.get(x, CANONICAL_LABEL_MAP.get(str(x).replace(' ', '_').replace('-', '_'), None))
    )

    # Filter out unmapped anomaly artifacts AND explicitly omit the zero-day holdout class
    zero_day_class_name = CONFIG['zero_day_class'] # 'Web_Brute_Force'

    valid_pool_df = aggregated_df[
        (aggregated_df['standardized_target_label'].notna()) &
        (aggregated_df['standardized_target_label'] != zero_day_class_name)
    ].copy()

    # Map the target text strings down into their numeric index integers
    valid_pool_df['mapped_class_index'] = valid_pool_df['standardized_target_label'].map(GLOBAL_CLASS_INDEX)

    # --- STRATIFIED COHORT ALLOCATION PASS ---
    unique_classes_isolated = valid_pool_df['standardized_target_label'].unique()
    print(f" -> Found {len(unique_classes_isolated)} matching trained structural classes in {target_dataset_tag} storage pool.")

    # Calculate uniform samples required per active available class layer
    samples_per_class = max(1, total_target_samples // len(unique_classes_isolated))

    stratified_sub_frames = []
    for cls_name in unique_classes_isolated:
        cls_slice = valid_pool_df[valid_pool_df['standardized_target_label'] == cls_name]

        # Pull up to samples_per_class from each available category
        use_replace = len(cls_slice) < samples_per_class
        cls_sample = cls_slice.sample(n=samples_per_class, replace=use_replace, random_state=RANDOM_STATE)
        stratified_sub_frames.append(cls_sample)

    final_stratified_df = pd.concat(stratified_sub_frames, ignore_index=True)

    # Reshuffle the final dataset array to simulate random live packet streams
    return final_stratified_df.sample(frac=1.0, random_state=RANDOM_STATE).copy()

# -------------------------------------------------------------------
# CROSS-DATASET INFERENCE PIPELINE INTERACTION LOOP
# -------------------------------------------------------------------
for target_dataset_tag in external_targets_pool:
    print(f"\n[Loader Core] Ingesting true raw datasets from Google Drive for: {target_dataset_tag}...")

    # Ingest stratified, multi-class sample cohorts directly from files (excluding Web_Brute_Force)
    true_external_df = aggregate_and_stratify_drive_directory(target_dataset_tag, total_target_samples=1000)

    # --- REAL-WORLD EXPLICIT FEATURE ALIGNMENT ENGINE ---
    transfer_data_dict = {}

    for feat_name in expected_features:
        target_csv_header = None

        # Map out-of-order column configurations via updated block 2 structures
        if target_dataset_tag == "CSE-CIC-IDS2018":
            if feat_name in FEATURE_MAP_CSE2018:
                target_csv_header = FEATURE_MAP_CSE2018[feat_name].strip()
        else: # CIC-IDS2017 Mode
            if feat_name in FEATURE_MAP_CIC:
                target_csv_header = FEATURE_MAP_CIC[feat_name].strip()

        # Check for explicit mappings or exact header strings in the columns
        resolved_header = None
        if target_csv_header and target_csv_header in true_external_df.columns:
            resolved_header = target_csv_header
        elif feat_name in true_external_df.columns:
            resolved_header = feat_name

        # --- POSITION-INVARIANT EXTRACTION & ZERO-FILL LAYER ---
        if resolved_header:
            transfer_data_dict[feat_name] = pd.to_numeric(true_external_df[resolved_header], errors='coerce').fillna(0).to_numpy()
        else:
            # Drop zero-fills for the 35+ advanced parameters not generated by CICFlowMeter
            transfer_data_dict[feat_name] = np.zeros(len(true_external_df))

    # Compile the dictionary directly into an ordered matrix format to preserve strict LightGBM column indexes
    transfer_matrix_builder = pd.DataFrame(transfer_data_dict, columns=expected_features)
    X_transfer_numpy = transfer_matrix_builder.to_numpy().astype(np.float32)
    y_transfer_numpy = true_external_df['mapped_class_index'].to_numpy().astype(np.int32)

    print(f" -> Successfully stabilized shape configuration array to: {X_transfer_numpy.shape}")
    print(f" -> Matrix multi-class distribution tracker:\n" + "-"*50)
    for idx, count in enumerate(np.bincount(y_transfer_numpy)):
        if count > 0:
            print(f"    Class {idx:02d} ({REVERSE_CLASS_INDEX[idx]:<20}): loaded {count} samples")
    print("-"*50)

    # --- REGISTER MATRICES IN GLOBAL MEMORY FOR DOWNSTREAM EVALUATIONS ---
    globals()['STAGE_9_CAPTURED_MATRICES'][target_dataset_tag] = (X_transfer_numpy.copy(), y_transfer_numpy.copy())

    # Run the shape-corrected evaluator function from Block 11
    transfer_metrics = evaluate_operational_environment(
        lgb_model, calibrators, train_signatures, TAU_P, TAU_S,
        X_transfer_numpy, y_transfer_numpy, f"Cross-Dataset Transfer: {target_dataset_tag}"
    )

    comprehensive_metrics_log.append(transfer_metrics)
    block_14_records.append(transfer_metrics)

# --- INSTANT CELL TERMINAL DISPLAY FORMATTING ENGINE ---
block_14_df = pd.DataFrame(block_14_records)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n" + "#" * 115)
print("             STAGE 9: REAL-WORLD SCHEMA-ALIGNED TRUE CROSS-DATASET TRANSFER RESULTS          ")
print("#" * 115)
print(block_14_df.to_string(index=False, formatters={
    'Global_Accuracy': '{:,.4f}'.format,
    'Macro_F1_Throughput': '{:,.4f}'.format,
    'Attack_Not_Missed_Rate_ANMR': '{:,.4f}'.format,
    'False_Block_Rate_FBR': '{:,.4f}'.format,
    'Escalation_Rate': '{:,.4f}'.format,
    'Latency_ms_per_packet': '{:,.3f} ms'.format,
    'Throughput_packets_per_sec': '{:,.1f} pps'.format,
    'RAM_Footprint_Overhead_MB': '{:,.2f} MB'.format
}))
print("#" * 115 + "\n")

In [ ]:
# BLOCK 15: STAGE 9 DIAGNOSTIC — CROSS-DATASET POLICY ROUTING AUDIT
import pandas as pd
import numpy as np
import shap

print("=" * 115)
print("          STAGE 9 DIAGNOSTIC: REAL-WORLD CROSS-DATASET POLICY ROUTING MULTI-AUDIT          ")
print("=" * 115)

# Extract cross-dataset matrices from global tracking slots
captured_map = globals().get('STAGE_9_CAPTURED_MATRICES', {})

if not captured_map:
    print("\n" + "!"*90)
    print(" [!] CHECKLIST ERROR: The diagnostic tracking register is empty.")
    print("     Please ensure you ran your updated Block 14 cell above before running this audit.")
    print("!"*90 + "\n")
else:
    print(f"[Engine Base] Successfully locked onto {len(captured_map)} active evaluation streams.\n")
    explainer_diag = shap.TreeExplainer(lgb_model)
    cross_dataset_report_records = []

    for env_tag, (X_diag, y_diag) in captured_map.items():
        print(f"[Auditing Engine] Mapping attribution geometries for: {env_tag}...")

        probs_diag = get_calibrated_probabilities(lgb_model, calibrators, X_diag)
        preds_diag = np.argmax(probs_diag, axis=1)
        shap_values_diag = explainer_diag.shap_values(X_diag)

        tier_1_benign_pass = 0
        tier_2a_canonical_block = 0
        tier_2b_drift_mitigation_block = 0
        tier_2c_autonomous_force_block = 0
        tier_3_administrative_escalation = 0
        total_malicious_flows = 0
        missed_attacks = 0

        for i in range(len(X_diag)):
            p_idx = preds_diag[i]
            p_name = REVERSE_CLASS_INDEX[p_idx]
            t_name = REVERSE_CLASS_INDEX[y_diag[i]]

            if t_name != 'Benign': total_malicious_flows += 1
            attack_confidence = 1.0 - probs_diag[i, GLOBAL_CLASS_INDEX['Benign']]

            if isinstance(shap_values_diag, np.ndarray) and shap_values_diag.ndim == 3:
                inst_shap = shap_values_diag[i, :, p_idx]
            elif isinstance(shap_values_diag, list):
                inst_shap = shap_values_diag[p_idx][i]
            else:
                inst_shap = shap_values_diag[i]

            norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)
            similarity = np.dot(norm_inst, train_signatures.get(p_name, np.zeros_like(norm_inst)))

            # --- RE-ENGINEERED MULTI-TIER ROUTING LOGIC (TAU_P=0.80, TAU_S=0.01 ALIGNED) ---
            if p_name == 'Benign' and attack_confidence < TAU_P and similarity >= TAU_S:
                tier_1_benign_pass += 1
                if t_name != 'Benign': missed_attacks += 1

            elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity >= TAU_S:
                tier_2a_canonical_block += 1

            elif p_name != 'Benign' and attack_confidence >= TAU_P and similarity >= (TAU_S * 0.5):
                tier_2b_drift_mitigation_block += 1

            elif p_name == 'Benign' and attack_confidence >= TAU_P and similarity < TAU_S:
                tier_2c_autonomous_force_block += 1

            else:
                if p_name == 'Benign' and t_name != 'Benign': missed_attacks += 1
                else: tier_3_administrative_escalation += 1

        cross_dataset_report_records.append({
            'Cross-Dataset Environment': env_tag,
            'Total Packets': len(X_diag),
            'Attacks Present': total_malicious_flows,
            'Tier 1 (Benign Pass)': tier_1_benign_pass,
            'Tier 2A (Canonical Block)': tier_2a_canonical_block,
            'Tier 2B (Drift Block)': tier_2b_drift_mitigation_block,
            'Tier 2C (Force Block)': tier_2c_autonomous_force_block,
            'Tier 3 (Escalated)': tier_3_administrative_escalation,
            'Silently Missed': missed_attacks
        })

    # --- DISPLAY HEAT TABLE ---
    cross_audit_df = pd.DataFrame(cross_dataset_report_records)
    print("\n" + "#" * 115)
    print("                CROSS-DATASET PERIMETER FIREWALL GATE PERFORMANCE LOGS (STAGE 9 AUDIT)               ")
    print("#" * 115)
    print(cross_audit_df.to_string(index=False))
    print("#" * 115 + "\n")

In [ ]:
# DIAGNOSTIC SUB-CELL: VERIFY STAGE 9 CROSS-DATASET CLASS DIVERSITY


captured_map = globals().get('STAGE_9_CAPTURED_MATRICES', {})

if not captured_map:
    print("[!] Error: Run your Block 14 cell first to populate memory registers.")
else:
    print("=" * 70)
    print("       EMERGENCY AUDIT: TRUE ATTACK CLASS DISTRIBUTION IN STAGE 9      ")
    print("=" * 70)
    for env_tag, (X_diag, y_diag) in captured_map.items():
        # Map integer array back to true readable class strings
        readable_labels = [REVERSE_CLASS_INDEX[ly] for ly in y_diag]
        distribution_series = pd.Series(readable_labels).value_counts()

        print(f"\n[Environment Matrix]: {env_tag}")
        print("-" * 50)
        print(distribution_series.to_string())
        print(f"Total Unique Classes Present: {len(distribution_series)}")
    print("=" * 70)

In [ ]:
# BLOCK 15: MASTER PRESENTATION ENGINE — CANONICAL BLUEPRINTS & SEPARATION GAP MANUSCRIPT LAYOUTS

from scipy.stats import spearmanr

print("=" * 115)
print("     LAUNCHING MASTER PRESENTATION ENGINE: CANONICAL SIGNATURES & SEPARATION MARGIN CONTINUUM      ")
print("=" * 115)

# ======================================================================================
# COMPONENT 1: RESTORED CANONICAL FEATURE IMPORTANCE BLUEPRINT EXPORT
# ======================================================================================
print("\n" + "#" * 115)
print("             MANUSCRIPT COMPONENT 1: CANONICAL TRAINING FEATURE IMPORTANCE BLUEPRINTS          ")
print("#" * 115 + "\n")

print("\\begin{table*}[htbp]")
print("\\centering")
print("\\caption{Game-Theoretic Local Attribution Space Feature Importance Blueprints Derived via L2-Normalized SHAP Target Trajectories}")
print("\\label{tab:canonical_shap_blueprints}")
print("\\begin{tabular}{llcccc}")
print("\\toprule")
print(f"{'Attack Profile':<16} & {'Top Features':<26} & {'Weight':<8} & {'Attack Profile':<16} & {'Top Features':<26} & {'Weight'} \\\\")
print("\\midrule")

classes_found = sorted(list(train_signatures.keys()))

# Process classes in pairs side-by-side to match the exact two-column table template
for idx in range(0, len(classes_found), 2):
    cls_left = classes_found[idx]
    cls_right = classes_found[idx+1] if idx+1 < len(classes_found) else None

    # Sort features by highest absolute impact vectors from the cached blueprints
    left_sorted_indices = np.argsort(np.abs(train_signatures[cls_left]))[::-1][:15]
    right_sorted_indices = np.argsort(np.abs(train_signatures[cls_right]))[::-1][:15] if cls_right else []

    for rank in range(15):
        # Format left column element
        l_idx = left_sorted_indices[rank]
        l_feat = expected_features[l_idx]
        l_val = train_signatures[cls_left][l_idx]
        l_sign = "+" if l_val >= 0 else ""
        escaped_cls_l = cls_left.replace('_', r'\_') if rank == 0 else ""
        escaped_feat_l = l_feat.replace('_', r'\_')

        left_str = f"{escaped_cls_l:<16} & {escaped_feat_l:<26} & {l_sign}{l_val:+.5f} &"

        # Format right column element
        if cls_right and rank < len(right_sorted_indices):
            r_idx = right_sorted_indices[rank]
            r_feat = expected_features[r_idx]
            r_val = train_signatures[cls_right][r_idx]
            r_sign = "+" if r_val >= 0 else ""
            escaped_cls_r = cls_right.replace('_', r'\_') if rank == 0 else ""
            escaped_feat_r = r_feat.replace('_', r'\_')

            right_str = f"{escaped_cls_r:<16} & {escaped_feat_r:<26} & {r_sign}{r_val:+.5f} \\\\"
        else:
            right_str = f"{'':<16} & {'':<26} & {'':<8} \\\\"

        print(f"{left_str} {right_str}")

    print("\\midrule")

print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table*}")


# ======================================================================================
# COMPONENT 2: GEOMETRIC SEPARATION GAP AUDIT EXPORT
# ======================================================================================
print("\n\n" + "#" * 115)
print("             MANUSCRIPT COMPONENT 2: ATTRIBUTION SPACE SEPARATION DISTINCTIVENESS MARGINS       ")
print("#" * 115 + "\n")

evaluation_pool = []

# Collect Stage 7 Drift Slices if active in global workspace
drift_mappings = [
    ('Train vs Host-Test', 'host_test_df'),
    ('Train vs Time-Test', 'time_test_df'),
    ('Train vs Hard-Test', 'hard_test_df')
]
for display_tag, df_var_name in drift_mappings:
    if df_var_name in globals() and globals()[df_var_name] is not None:
        evaluation_pool.append((display_tag, globals()[df_var_name]))

# Collect Stage 9 Cross-Dataset Streams if active
captured_map = globals().get('STAGE_9_CAPTURED_MATRICES', {})
for dataset_name, (X_mat, y_mat) in captured_map.items():
    df_temp = pd.DataFrame(X_mat, columns=expected_features)
    df_temp['label'] = [REVERSE_CLASS_INDEX[ly] for ly in y_mat]
    evaluation_pool.append((f"Train vs {dataset_name}", df_temp))

if not evaluation_pool:
    print("[!] Execution Flatline: No active drift matrices found in memory namespaces to complete Component 2.")
else:
    explainer_diag = shap.TreeExplainer(lgb_model)
    latex_rows = []

    for comp_tag, df_slice in evaluation_pool:
        X_diag = df_slice[expected_features].to_numpy()
        y_diag_strings = df_slice['label'].to_numpy()

        raw_preds = lgb_model.predict(X_diag)
        preds_diag = np.argmax(raw_preds, axis=1) if raw_preds.ndim > 1 else raw_preds
        shap_values_diag = explainer_diag.shap_values(X_diag)

        class_metrics = {}

        for i in range(len(X_diag)):
            true_class = y_diag_strings[i]
            pred_idx = preds_diag[i]

            if isinstance(shap_values_diag, np.ndarray) and shap_values_diag.ndim == 3:
                inst_shap = shap_values_diag[i, :, pred_idx]
            elif isinstance(shap_values_diag, list):
                inst_shap = shap_values_diag[pred_idx][i]
            else:
                inst_shap = shap_values_diag[i]

            norm_inst = inst_shap / (np.linalg.norm(inst_shap) + 1e-9)

            if true_class not in train_signatures: continue
            if true_class not in class_metrics:
                class_metrics[true_class] = {'same_scores': [], 'other_scores': []}

            # Spearman alignment against its own true signature blueprint
            ref_sig_same = train_signatures[true_class]
            rho_same, _ = spearmanr(norm_inst, ref_sig_same)
            if np.isnan(rho_same): rho_same = 0.0

            # Spearman cross-correlation against alternative counter-signatures
            other_rhos = []
            for alt_class, ref_sig_alt in train_signatures.items():
                if alt_class == true_class: continue
                rho_alt, _ = spearmanr(norm_inst, ref_sig_alt)
                if not np.isnan(rho_alt):
                    other_rhos.append(rho_alt)

            max_rho_other = np.max(other_rhos) if other_rhos else 0.0
            class_metrics[true_class]['same_scores'].append(rho_same)
            class_metrics[true_class]['other_scores'].append(max_rho_other)

        env_classes = sorted(list(class_metrics.keys()))
        for c_idx, cls in enumerate(env_classes):
            mean_same = np.mean(class_metrics[cls]['same_scores']) if class_metrics[cls]['same_scores'] else 0.0
            mean_other = np.mean(class_metrics[cls]['other_scores']) if class_metrics[cls]['other_scores'] else 0.0
            gap = mean_same - mean_other
            interpretation = "stable/distinct" if gap >= 0.15 else "weak/overlapping"

            escaped_cls = cls.replace('_', r'\_')
            escaped_tag = comp_tag.replace('_', r'\_')

            if c_idx == 0:
                prefix_str = f"\\multirow{{{len(env_classes)}}}{{*}}{{{escaped_tag}}}"
            else:
                prefix_str = ""

            latex_rows.append(
                f"{prefix_str:<32} & {escaped_cls:<24} & {mean_same:.4f} & {mean_other:.4f} & {gap:+.4f} & {interpretation} \\\\"
            )
        latex_rows.append("\\midrule")

    print("\\begin{table*}[htbp]")
    print("\\centering")
    print("\\caption{Game-Theoretic Local Attribution Space Separation and Distinctiveness Margin Analysis Across Domain Drift Boundaries}")
    print("\\label{tab:shap_spearman_gap_analysis}")
    print("\\begin{tabular}{llcccc}")
    print("\\toprule")
    print(f"{'Comparison':<32} & {'Attack':<24} & {'Same':<6} & {'Other':<6} & {'Gap':<7} & {'Interpretation'} \\\\")
    print("\\midrule")

    for row in latex_rows[:-1]: # Strip trailing midrule cleanly
        print(row)

    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table*}")
    print("\n" + "#" * 115 + "\n")

In [ ]:
# BLOCK 16: PERSISTENT DRIVE MASTER EXPORT CONSOLIDATOR & SYSTEM GRAPH GENERATOR
print("[System Consolidator] Compiling master operational matrices...")
master_report_df = pd.DataFrame(comprehensive_metrics_log)
master_report_df.to_csv(os.path.join(RESULTS_DIR, 'tables', 'comprehensive_perfected_pipeline_metrics.csv'), index=False)

try:
    drive_destination_path = os.path.join(CONFIG['results_dir'], 'comprehensive_perfected_pipeline_metrics.csv')
    master_report_df.to_csv(drive_destination_path, index=False)
    print("[Google Drive Master Synced] Final tables written cleanly to project directory path at:", drive_destination_path)
except Exception as e:
    print("[Google Drive Sync Alert] Active cloud mount connection trace invisible. Table data preserved in local instance cache runtime workspace.")

plt.figure(figsize=(11, 5.5))
plt.barh(master_report_df['Target_Environment'], master_report_df['Attack_Not_Missed_Rate_ANMR'], color='navy', edgecolor='black', height=0.45)
plt.xlabel('Attack-Not-Missed Rate (ANMR)')
plt.title('Heuristic Safety Gate Containment Capacity Across Core Evaluation Spaces')
plt.xlim(0, 1.05)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'safety_gate_anmr_generalization.png'), dpi=300)
plt.close()

print("\n=========================================================================================")
print("               ALL EXPERIMENTAL PILLARS RE-ARCHITECTED SUCCESSFULLY FOR RESUBMISSION       ")
print("=========================================================================================")
print(master_report_df.to_string(index=False))

In [ ]:
# ===================================================================
# FINAL CHAPTER: COMPLETE COMPREHENSIVE MULTI-SEED REPLICABILITY ENGINE (Dynamic Context Strategy)
# ===================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import glob
import os
import gc
from scipy.stats import spearmanr
from sklearn.calibration import IsotonicRegression
from sklearn.metrics import accuracy_score, f1_score
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print("[Master Validation Engine] Initiating complete 6-pillar multi-seed verification harness...")

# Strict parameter locks optimized from validation passes
SEEDS = [42, 123, 456]
FINAL_TAU_P = 0.7400
FINAL_TAU_S = 0.0100
CLASS_COUNT = 12
BENIGN_IDX = GLOBAL_CLASS_INDEX['Benign']
ZERO_DAY_IDX = GLOBAL_CLASS_INDEX[CONFIG['zero_day_class']]

# Extract historical profiles from the training dataset for feature imputation
training_feature_means = train_df[expected_features].mean().to_dict()
REVERSE_FEATURE_MAP_CIC = {v: k for k, v in FEATURE_MAP_CIC.items()}

# 1. Establish the internal static baseline test tracking frameworks
base_environments = {
    "Host-Test (Topological Shift)": host_test_df,
    "Time-Test (Temporal Drift)": time_test_df,
    "Hard-Test (Combined Shift)": hard_test_df,
    "Zero-Day Evolution Holdout": df_raw[df_raw['label'] == CONFIG['zero_day_class']].copy()
}

cross_dataset_tags = ["CIC-IDS2017", "CSE-CIC-IDS2018"]
all_6_pillars = list(base_environments.keys()) + [f"Cross-Dataset Transfer: {t}" for t in cross_dataset_tags]

# Master log ledger to store calculations across runs for standard deviation rendering
multi_seed_telemetry_registry = {
    env: {"ANMR": [], "FBR": [], "Global_Acc": [], "Macro_F1": []} for env in all_6_pillars
}

def row_l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms > 1e-9)

def dynamic_stratified_loader_loop(target_dataset_tag, loop_seed, total_target_samples=1000):
    """Dynamically parses and samples unique multi-class cohorts from raw Drive paths on every seed iteration."""
    dir_key = 'cic2017_dir' if target_dataset_tag == "CIC-IDS2017" else 'cse2018_dir'
    target_dir = CONFIG[dir_key]
    csv_files = glob.glob(os.path.join(target_dir, "*.csv"))

    collected_chunks = []
    for file_path in csv_files:
        try:
            chunk_df = pd.read_csv(file_path, low_memory=False)
            if not chunk_df.empty:
                chunk_df.columns = [str(c).strip() for c in chunk_df.columns]
                collected_chunks.append(chunk_df)
        except Exception:
            continue

    aggregated_df = pd.concat(collected_chunks, ignore_index=True)
    label_col_marker = [c for c in aggregated_df.columns if c.lower() == 'label'][0]

    raw_labels_cleaned = aggregated_df[label_col_marker].astype(str).str.strip().str.replace(r'[^\x00-\x7F]+', '_', regex=True)
    aggregated_df['standardized_target_label'] = raw_labels_cleaned.map(
        lambda x: CANONICAL_LABEL_MAP.get(x, CANONICAL_LABEL_MAP.get(str(x).replace(' ', '_').replace('-', '_'), None))
    )

    zero_day_class_name = CONFIG['zero_day_class']
    valid_pool_df = aggregated_df[
        (aggregated_df['standardized_target_label'].notna()) &
        (aggregated_df['standardized_target_label'] != zero_day_class_name)
    ].copy()

    valid_pool_df['mapped_class_index'] = valid_pool_df['standardized_target_label'].map(GLOBAL_CLASS_INDEX)
    unique_classes_isolated = valid_pool_df['standardized_target_label'].unique()

    samples_per_class = max(1, total_target_samples // len(unique_classes_isolated))

    stratified_sub_frames = []
    for cls_name in unique_classes_isolated:
        cls_slice = valid_pool_df[valid_pool_df['standardized_target_label'] == cls_name]
        use_replace = len(cls_slice) < samples_per_class
        # CRITICAL FIX: Link the row sampler directly to the loop seed
        cls_sample = cls_slice.sample(n=samples_per_class, replace=use_replace, random_state=loop_seed)
        stratified_sub_frames.append(cls_sample)

    final_stratified_df = pd.concat(stratified_sub_frames, ignore_index=True)
    return final_stratified_df.sample(frac=1.0, random_state=loop_seed).copy()

# -------------------------------------------------------------------
# MASTER SEED INITIALIZATION EXECUTION LOOP
# -------------------------------------------------------------------
for current_seed in SEEDS:
    print(f"\n" + "="*90 + f"\n[Harness Run] Executing full pipeline loop for fresh context Seed: {current_seed}\n" + "="*90)
    np.random.seed(current_seed)

    X_train_mat = train_df[expected_features].values.astype(np.float32)
    y_train_mat = train_df['label'].map(GLOBAL_CLASS_INDEX).values.astype(np.int32)
    X_val_mat = val_df[expected_features].values.astype(np.float32)
    y_val_mat = val_df['label'].map(GLOBAL_CLASS_INDEX).values.astype(np.int32)

    # Train structural LightGBM model utilizing the active loop random state seed
    train_data = lgb.Dataset(X_train_mat, label=y_train_mat)
    val_data = lgb.Dataset(X_val_mat, label=y_val_mat, reference=train_data)

    params = {
        'objective': 'multiclass', 'num_class': CLASS_COUNT, 'metric': 'multi_logloss',
        'learning_rate': 0.05, 'num_leaves': 32, 'random_state': current_seed, 'verbose': -1
    }
    loop_model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[val_data], callbacks=[lgb.early_stopping(10, verbose=False)])

    # Fit multi-channel piecewise Isotonic Regression over the loop raw probabilities
    val_raw_preds = loop_model.predict(X_val_mat)
    loop_calibrators = {}
    for c in range(CLASS_COUNT):
        ir = IsotonicRegression(out_of_bounds='clip')
        target_binary = (y_val_mat == c).astype(int)
        if len(np.unique(target_binary)) < 2:
            target_binary = np.append(target_binary, [0, 1])
            val_raw_preds_padded = np.append(val_raw_preds[:, c], [0.0, 1.0])
            ir.fit(val_raw_preds_padded, target_binary)
        else:
            ir.fit(val_raw_preds[:, c], target_binary)
        loop_calibrators[c] = ir

    def get_calibrated_probabilities_loop(model, idx_calibrators, X_mat):
        raw = model.predict(X_mat)
        calibrated = np.zeros_like(raw)
        for cls_idx in range(CLASS_COUNT):
            calibrated[:, cls_idx] = idx_calibrators[cls_idx].transform(raw[:, cls_idx])
        sums = calibrated.sum(axis=1, keepdims=True)
        sums[sums == 0] = 1.0
        return calibrated / sums

    # Generate seed-specific game-theoretic signature blueprints
    loop_explainer = shap.TreeExplainer(loop_model)
    loop_signatures = {}
    for class_idx in range(CLASS_COUNT):
        if class_idx == ZERO_DAY_IDX: continue
        class_rows = np.where(y_train_mat == class_idx)[0]
        if len(class_rows) == 0: continue
        np.random.seed(current_seed)
        sampled_rows = np.random.choice(class_rows, size=min(len(class_rows), 40), replace=False)
        raw_shap = loop_explainer.shap_values(X_train_mat[sampled_rows])
        class_shap = raw_shap[class_idx] if isinstance(raw_shap, list) else (raw_shap[:, :, class_idx] if len(raw_shap.shape) == 3 else raw_shap)
        norm_shap = row_l2_normalize(class_shap)
        mean_vector = np.mean(norm_shap, axis=0)
        loop_signatures[class_idx] = mean_vector / (np.linalg.norm(mean_vector) + 1e-9)

    # -------------------------------------------------------------------
    # PERFORMANCE STREAM EVALUATION ACROSS ALL 6 EXPERIMENTAL PILLARS
    # -------------------------------------------------------------------
    for env_name in all_6_pillars:
        if "Cross-Dataset" in env_name:
            tag = "CIC-IDS2017" if "CIC-IDS2017" in env_name else "CSE-CIC-IDS2018"

            # CRITICAL IMPLEMENTATION FIX: Dynamic loading linked securely to current run seed
            true_ext_df = dynamic_stratified_loader_loop(tag, current_seed, total_target_samples=1000)

            transfer_data_dict = {}
            for feat_name in expected_features:
                target_csv_header = None
                if tag == "CSE-CIC-IDS2018":
                    if feat_name in FEATURE_MAP_CSE2018: target_csv_header = FEATURE_MAP_CSE2018[feat_name].strip()
                else:
                    if feat_name in FEATURE_MAP_CIC: target_csv_header = FEATURE_MAP_CIC[feat_name].strip()

                resolved_header = None
                if target_csv_header and target_csv_header in true_ext_df.columns: resolved_header = target_csv_header
                elif feat_name in true_ext_df.columns: resolved_header = feat_name

                if resolved_header:
                    transfer_data_dict[feat_name] = pd.to_numeric(true_ext_df[resolved_header], errors='coerce').fillna(0).to_numpy()
                else:
                    transfer_data_dict[feat_name] = np.full(len(true_ext_df), training_feature_means[feat_name])

            transfer_matrix_builder = pd.DataFrame(transfer_data_dict, columns=expected_features)
            X_eval = transfer_matrix_builder.to_numpy().astype(np.float32)
            y_eval = true_ext_df['mapped_class_index'].to_numpy().astype(np.int32)
        else:
            env_df = base_environments[env_name].copy()
            X_eval_list = []
            for feat in expected_features:
                if feat in env_df.columns:
                    X_eval_list.append(pd.to_numeric(env_df[feat], errors='coerce').fillna(0).values)
                else:
                    X_eval_list.append(np.zeros(len(env_df)))
            X_eval = np.column_stack(X_eval_list).astype(np.float32)
            y_eval = env_df['label'].map(GLOBAL_CLASS_INDEX).fillna(-1).values.astype(np.int32) if 'label' in env_df.columns else np.full(len(env_df), -1, dtype=np.int32)

        cal_probs = get_calibrated_probabilities_loop(loop_model, loop_calibrators, X_eval)
        y_pred = np.argmax(cal_probs, axis=1)
        attack_conf = 1.0 - cal_probs[:, BENIGN_IDX]
        raw_eval_shap = loop_explainer.shap_values(X_eval)

        m_missed, b_false, total_malicious, total_benign = 0, 0, 0, 0

        for i in range(len(y_pred)):
            pred_c, true_c = y_pred[i], y_eval[i]
            is_malicious = (true_c != BENIGN_IDX)
            if is_malicious: total_malicious += 1
            else: total_benign += 1

            instance_shap = raw_eval_shap[pred_c][i] if isinstance(raw_eval_shap, list) else (raw_eval_shap[i, :, pred_c] if len(raw_eval_shap.shape) == 3 else raw_eval_shap[i])
            norm_instance_shap = instance_shap / (np.linalg.norm(instance_shap) + 1e-9)

            if pred_c in loop_signatures:
                rho_same, _ = spearmanr(norm_instance_shap, loop_signatures[pred_c])
                if np.isnan(rho_same): rho_same = 0.0
                rho_other = max([spearmanr(norm_instance_shap, alt_sig)[0] for alt_c, alt_sig in loop_signatures.items() if alt_c != pred_c])
                separation_gap = rho_same - rho_other
            else:
                separation_gap = -1.0

            if pred_c == BENIGN_IDX and attack_conf[i] < FINAL_TAU_P and separation_gap >= FINAL_TAU_S:
                action = "ALLOW"
            elif pred_c != BENIGN_IDX and attack_conf[i] >= FINAL_TAU_P and separation_gap >= (0.5 * FINAL_TAU_S):
                action = "BLOCK"
            else:
                action = "ESCALATE"

            if action == "ALLOW" and is_malicious: m_missed += 1
            if action != "ALLOW" and not is_malicious: b_false += 1

        acc = accuracy_score(y_eval, y_pred) if -1 not in y_eval else 0.0
        f1 = f1_score(y_eval, y_pred, average='macro') if -1 not in y_eval else 0.0

        multi_seed_telemetry_registry[env_name]["ANMR"].append((total_malicious - m_missed) / (total_malicious + 1e-9))
        multi_seed_telemetry_registry[env_name]["FBR"].append(b_false / (total_benign + 1e-9))
        multi_seed_telemetry_registry[env_name]["Global_Acc"].append(acc)
        multi_seed_telemetry_registry[env_name]["Macro_F1"].append(f1)

    del loop_model, loop_explainer, loop_signatures, X_train_mat, X_val_mat
    gc.collect()

# -------------------------------------------------------------------
# FINAL TELEMETRY PRINT PASS FOR JOURNAL MANUSCRIPT SECTIONS
# -------------------------------------------------------------------
print("\n" + "="*95 + "\nFINAL CONSOLIDATED MULTI-SEED REPLICABILITY MATRIX (\mu \pm \sigma)\n" + "="*95)
for env_name, metrics in multi_seed_telemetry_registry.items():
    print(f"Operational Horizon Pillar: {env_name}")
    print(f"  -> Global Accuracy    : {np.mean(metrics['Global_Acc']):.4f} +/- {np.std(metrics['Global_Acc']):.4f}")
    print(f"  -> Macro F1-Score     : {np.mean(metrics['Macro_F1']):.4f} +/- {np.std(metrics['Macro_F1']):.4f}")
    print(f"  -> Safety Gate ANMR   : {np.mean(metrics['ANMR']):.4f} +/- {np.std(metrics['ANMR']):.4f}")
    print(f"  -> Safety Gate FBR    : {np.mean(metrics['FBR']):.4f} +/- {np.std(metrics['FBR']):.4f}")
    print("-" * 95)

In [ ]:
# ===================================================================
# MULTI-DOMAIN CORRELATION INTERROGATOR: HEADER SCHEMAS ALIGNMENT
# ===================================================================
import os
import glob
import pandas as pd

def audit_all_dataset_schemas():
    print("=" * 100)
    print("           LAUNCHING TRIPLE-DOMAIN LOGICAL SCHEMA ALIGNMENT DISCOVERY")
    print("=" * 100)

    # 1. Inspect the Base Clean Invariant Features from your Active Pipeline Memory
    if 'df_clean' in locals() or 'df_clean' in globals():
        source_df = locals().get('df_clean', globals().get('df_clean'))
        print(f"✅ [Domain 1] BCCC-CIC-IDS-2017 Memory Blueprint ({len(expected_features)} Invariant Attributes Locked)")
        print(f"Sample Features: {expected_features[:8]} ...")
        print("-" * 100)
        for idx, col in enumerate(sorted(expected_features), 1):
            print(f"  [{idx:03d}] Invariant Token: '{col}'")
    else:
        print("❌ [Domain 1 Error] 'df_clean' not detected in active environment memory.")

    print("\n" + "=" * 100)

    # 2. Inspect the Raw Directory Headers for CIC-IDS2017 from Google Drive
    cic2017_path = CONFIG.get('cic2017_dir', None)
    if cic2017_path and os.path.exists(cic2017_path):
        cic_files = [f for f in os.listdir(cic2017_path) if f.endswith('.csv')]
        if cic_files:
            sample_cic_file = os.path.join(cic2017_path, cic_files[0])
            print(f"✅ [Domain 2] Raw Disk Headers for Target: CIC-IDS2017")
            print(f"Inspecting Source File Asset: {cic_files[0]}")
            print("-" * 100)
            try:
                cic_headers = list(pd.read_csv(sample_cic_file, nrows=0).columns)
                for idx, col in enumerate(sorted(cic_headers), 1):
                    print(f"  [{idx:02d}] Raw File Header: '{col}'")
            except Exception as e:
                print(f"Crash parsing CIC-IDS2017 headers: {str(e)}")
        else:
            print("❌ [Domain 2 Error] No CSV data assets found in your CIC-IDS2017 Drive folder.")
    else:
        print(f"❌ [Domain 2 Error] CIC-IDS2017 Path is missing or unreachable: {cic2017_path}")

    print("\n" + "=" * 100)

    # 3. Inspect the Raw Directory Headers for CSE-CIC-IDS2018 from Google Drive
    cse2018_path = CONFIG.get('cse2018_dir', None)
    if cse2018_path and os.path.exists(cse2018_path):
        cse_files = [f for f in os.listdir(cse2018_path) if f.endswith('.csv')]
        if cse_files:
            sample_cse_file = os.path.join(cse2018_path, cse_files[0])
            print(f"✅ [Domain 3] Raw Disk Headers for Target: CSE-CIC-IDS2018")
            print(f"Inspecting Source File Asset: {cse_files[0]}")
            print("-" * 100)
            try:
                cse_headers = list(pd.read_csv(sample_cse_file, nrows=0).columns)
                for idx, col in enumerate(sorted(cse_headers), 1):
                    print(f"  [{idx:02d}] Raw File Header: '{col}'")
            except Exception as e:
                print(f"Crash parsing CSE-CIC-IDS2018 headers: {str(e)}")
        else:
            print("❌ [Domain 3 Error] No CSV data assets found in your CSE-CIC-IDS2018 Drive folder.")
    else:
        print(f"❌ [Domain 3 Error] CSE-CIC-IDS2018 Path is missing or unreachable: {cse2018_path}")
    print("=" * 100)

# Run the alignment scanner
audit_all_dataset_schemas()